## Hitta fellänkar WD -> SPA för Tvåkammar-riksdagen 1867–1970
* [#56](https://github.com/salgo60/spa2Commons/issues/56)
* denna Notebook [56_WD_SPA_check](https://github.com/salgo60/spa2Commons/blob/main/Notebook/56_WD_SPA_check.ipynb)

In [1]:
from datetime import datetime
start_time  = datetime.now()
print("Last run: ", start_time)

Last run:  2026-03-08 12:46:19.712706


In [ ]:
# ==========================================
# SPA reference validation report
# ==========================================

import requests,time
import pandas as pd
import datetime
from SPARQLWrapper import SPARQLWrapper, JSON
from tqdm import tqdm


# ------------------------------------------
# Metadata
# ------------------------------------------

now = datetime.datetime.utcnow()

report_date = now.strftime("%Y-%m-%d")
filename_date = now.strftime("%y%m%d")

report_filename = f"spa_report_{filename_date}.html"

issue_url = "https://github.com/salgo60/spa2Commons/issues/56"

print("Report file:", report_filename)
print("Run date:", report_date)


# ------------------------------------------
# SPARQL query
# ------------------------------------------

query = """
SELECT DISTINCT
?person
?personLabel
?statement
?SPAid
WHERE {

  VALUES ?source { wd:Q110346241 }

  ?person p:P1343 ?statement .
  ?statement ps:P1343 ?source .

  OPTIONAL {
      ?statement prov:wasDerivedFrom [
        pr:P4819 ?SPAid
      ]
  }

  SERVICE wikibase:label { bd:serviceParam wikibase:language "sv,en". }

}
ORDER BY ?personLabel
"""


# ------------------------------------------
# SPARQL session with User-Agent
# ------------------------------------------

sparql = SPARQLWrapper(
    "https://query.wikidata.org/sparql",
    agent="SPA reference checker (salgo60@msn.com)"
)

sparql.setQuery(query)
sparql.setReturnFormat(JSON)

print("Running SPARQL query...")
results = sparql.query().convert()


# ------------------------------------------
# Convert to dataframe
# ------------------------------------------

rows = []

for r in results["results"]["bindings"]:

    spaid = r.get("SPAid", {}).get("value")

    if not spaid:
        continue

    rows.append({

        "person": r["person"]["value"],
        "label": r["personLabel"]["value"],
        "statement": r["statement"]["value"],
        "SPAid": spaid
    })

df = pd.DataFrame(rows)

print("Records with SPAID:", len(df))


# ------------------------------------------
# SPA API check
# ------------------------------------------

def check_spa(spaid):
    # source → Facts → PortraitCatalog

    url = f"https://portrattarkiv.se/endpoints/portraits.php?id={spaid}"

    try:

        r = requests.get(url, timeout=20)
        data = r.json()

        catalogs = data.get("source", {}).get("Facts", {}).get("PortraitCatalog", [])

        correct = any(
            "tvåkammar-riksdagen 1867" in c.lower()
            for c in catalogs
        )

        return catalogs, correct

    except Exception as e:

        return [str(e)], False 
        
        
def check_spa_debug(spaid):

    url = f"https://portrattarkiv.se/endpoints/portraits.php?id={spaid}"

    r = requests.get(url, timeout=20)

    data = r.json()

    print("\nSPA ID:", spaid)
    print("PortraitCatalog RAW:")
    print(data.get("PortraitCatalog"))

    return data
# ------------------------------------------
# Validate SPA IDs
# ------------------------------------------

results_list = []

print("Checking SPA API...")

for _, row in tqdm(df.iterrows(), total=len(df)):
    #check_spa_debug(row["SPAid"])
    catalog, correct = check_spa(row["SPAid"])

    qid = row["person"].split("/")[-1]
    time.sleep(0.2)  
    results_list.append({

        "Person": row["label"],
        "QID": qid,
        "StatementURI": row["statement"],
        "SPAid": row["SPAid"],
        "Catalog": catalog,
        "Correct": correct

    })

check_df = pd.DataFrame(results_list)


# ------------------------------------------
# Create links
# ------------------------------------------

def api_link(spaid):

    url = f"https://portrattarkiv.se/endpoints/portraits.php?id={spaid}"

    return f'<a href="{url}">API</a>'

def wd_link(qid):

    return f"https://www.wikidata.org/wiki/{qid}"

def statement_link(statement_uri):

    return statement_uri

def spa_link(spaid):

    return f"https://portrattarkiv.se/details/{spaid}"


check_df["Wikidata"] = check_df["QID"].apply(
    lambda q: f'<a href="{wd_link(q)}">{q}</a>'
)

check_df["SPA"] = check_df["SPAid"].apply(
    lambda s: f'<a href="{spa_link(s)}">{s}</a>'
)

check_df["Statement"] = check_df["StatementURI"].apply(
    lambda s: f'<a href="{statement_link(s)}">statement</a>'
)

check_df["API"] = check_df["SPAid"].apply(api_link)


# ------------------------------------------
# Filter errors
# ------------------------------------------

errors = check_df[check_df["Correct"] == False]

print("Possible errors:", len(errors))


# ------------------------------------------
# Generate QuickStatements
# ------------------------------------------

def extract_statement_id(uri):

    return uri.split("/")[-1]

errors["StatementID"] = errors["StatementURI"].apply(extract_statement_id)

qs_lines = []

for _, r in errors.iterrows():

    qs_lines.append(
        f'-{r["StatementID"]} P4819 "{r["SPAid"]}"'
    )

quickstatements = "\n".join(qs_lines)


# ------------------------------------------
# HTML report
# ------------------------------------------

html = f"""
<html>
<head>
<meta charset="utf-8">

<title>SPA reference validation</title>

<style>

body {{
font-family: Arial;
margin: 40px;
}}

table {{
border-collapse: collapse;
width: 100%;
}}

td, th {{
border: 1px solid #ddd;
padding: 6px;
}}

th {{
background: #f2f2f2;
}}

</style>

</head>

<body>

<h1>SPA reference validation report</h1>

<p>
<b>Run date:</b> {report_date}
</p>

<p>
<b>Source property:</b>
<a href="https://www.wikidata.org/wiki/Property:P1343">
P1343 stated in
</a>
</p>

<p>
<b>Issue:</b>
<a href="{issue_url}">
spa2Commons issue 56
</a>
</p>

<p>
<b>Total checked:</b> {len(check_df)}<br>
<b>Possible errors:</b> {len(errors)}
</p>

<h2>Suspected incorrect SPA references</h2>

{errors[[
    "Person",
    "Wikidata",
    "Statement",
    "SPA",
    "API",
    "Catalog"
]].to_html(escape=False,index=False)}

<h2>QuickStatements</h2>

<p>Remove incorrect SPA reference (P4819)</p>

<pre>
{quickstatements}
</pre>

</body>
</html>
"""


# ------------------------------------------
# Save report
# ------------------------------------------

with open(report_filename, "w", encoding="utf-8") as f:

    f.write(html)

print("Report saved:", report_filename)

/var/folders/fd/md6r13sj0wsbg_6_xl160d300000gn/T/ipykernel_32720/3114045660.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.datetime.utcnow()


Report file: spa_report_260308.html
Run date: 2026-03-08
Running SPARQL query...
Records with SPAID: 3590
Checking SPA API...


 46%|██████████████████                     | 1664/3590 [11:20<13:07,  2.45it/s]

In [ ]:
from datetime import datetime
end = datetime.now()
print("Ended: ", end) 
print('Time elapsed (hh:mm:ss.ms) {}'.format(datetime.now() - start_time))